In [9]:
import pandas as pd
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, Bidirectional, LSTM, TextVectorization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [10]:
all_files = glob.glob("*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {all_files}")

df.info()
df.head()

Total rows: 17864
Dari 9 file: ['linkedin_jobs_20260418_215155_webdev.csv', 'linkedin_jobs_20260426_002316.csv', 'linkedin_jobs_20260430_001046.csv', 'linkedin_jobs_20260505_063507.csv', 'jobstreet_cleaned terbaru.csv', 'linkedin_jobs_20260416_223428_datscien.csv', 'jobstreet_jobs_20260421_164240.csv', 'jobstreet_final_20260509_173743.csv', 'linkedin_jobs_20260421_215330_softdev.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17864 entries, 0 to 17863
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17662 non-null  float64
 1   title             17662 non-null  object 
 2   company           17657 non-null  object 
 3   location          17598 non-null  object 
 4   job_description   17830 non-null  object 
 5   job_url           17662 non-null  object 
 6   search_role       17864 non-null  object 
 7   search_location   17864 non-null  object 
 8   extracted_skills  17864 non-

,id,title,company,location,job_description,job_url,search_role,search_location,extracted_skills,skills_count,scraped_at,salary,search_role_raw,job_level,location_raw,salary_display,salary_min,salary_max,salary_avg
0,4.392399e+09,Power BI Development,TOG Indonesia,"South Jakarta, Jakarta, Indonesia (On-site)",About the job\n\nResponsibilities\n\nMigrate e...,https://www.linkedin.com/jobs/view/4392398809/...,Web Developer,Indonesia,"['power bi', 'r', 'sql', 'gcp']",4,2026-04-18T20:31:41.459604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.403825e+09,Python Developer | Remote,Crossing Hurdles,Indonesia (Remote),About the job\n\nPosition: Python Developer\n\...,https://www.linkedin.com/jobs/view/4403825344/...,Web Developer,Indonesia,"['r', 'python', 'scala', 'git']",4,2026-04-18T20:31:46.381431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.400362e+09,"Software Engineer (Surabaya, Indonesia)",camLine | Elisa Industriq,"Surabaya, East Java, Indonesia (On-site)",About the job\n\nJob Description :\n\n\nInvolv...,https://www.linkedin.com/jobs/view/4400362349/...,Web Developer,Indonesia,"['r', 'java']",2,2026-04-18T20:31:51.324181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.370637e+09,Business Intelligence - Engineering Intern (Da...,Shopee,"Jakarta, Indonesia (On-site)",About the job\n\nJob Description:\n\nWe are lo...,https://www.linkedin.com/jobs/view/4370636535/...,Web Developer,Indonesia,"['r', 'kubernetes', 'python', 'sql']",4,2026-04-18T20:31:56.211140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.299316e+09,Data Analyst / Data Engineer – SCG Indonesia,SCG,"Jakarta, Indonesia (On-site)",About the job\n\nAbout SCG in Indonesia\n\nWit...,https://www.linkedin.com/jobs/view/4299315688/...,Web Developer,Indonesia,"['power bi', 'r', 'tableau', 'python', 'sql', ...",6,2026-04-18T20:32:01.140377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(f"Jumlah unique role: {df['search_role'].nunique()}")
print()
print(df['search_role'].value_counts())

Jumlah unique role: 24

search_role
Data Engineer           2376
Web Developer           1713
System Analyst          1685
Data Scientist          1491
Software Engineer       1351
Backend Developer       1005
Developer               1000
Full stack Developer    1000
Software Developer      1000
Data Analyst             978
Software                 885
IT Support               770
Frontend Developer       702
IT                       571
DevOps Engineer          405
Business Analyst         294
Full Stack Developer     112
Fullstack Developer      110
Programmer               108
Ai Engineer               83
ML Engineer               83
software                  64
Business Development      56
Other                     22
Name: count, dtype: int64


In [12]:
df = df.drop_duplicates()
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")

total duplicated columns : 0


In [13]:
df.head()

,id,title,company,location,job_description,job_url,search_role,search_location,extracted_skills,skills_count,scraped_at,search_role_raw,job_level,location_raw,salary_display,salary_min,salary_max,salary_avg
0,4.392399e+09,Power BI Development,TOG Indonesia,"South Jakarta, Jakarta, Indonesia (On-site)",About the job\n\nResponsibilities\n\nMigrate e...,https://www.linkedin.com/jobs/view/4392398809/...,Web Developer,Indonesia,"['power bi', 'r', 'sql', 'gcp']",4,2026-04-18T20:31:41.459604,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.403825e+09,Python Developer | Remote,Crossing Hurdles,Indonesia (Remote),About the job\n\nPosition: Python Developer\n\...,https://www.linkedin.com/jobs/view/4403825344/...,Web Developer,Indonesia,"['r', 'python', 'scala', 'git']",4,2026-04-18T20:31:46.381431,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.400362e+09,"Software Engineer (Surabaya, Indonesia)",camLine | Elisa Industriq,"Surabaya, East Java, Indonesia (On-site)",About the job\n\nJob Description :\n\n\nInvolv...,https://www.linkedin.com/jobs/view/4400362349/...,Web Developer,Indonesia,"['r', 'java']",2,2026-04-18T20:31:51.324181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.370637e+09,Business Intelligence - Engineering Intern (Da...,Shopee,"Jakarta, Indonesia (On-site)",About the job\n\nJob Description:\n\nWe are lo...,https://www.linkedin.com/jobs/view/4370636535/...,Web Developer,Indonesia,"['r', 'kubernetes', 'python', 'sql']",4,2026-04-18T20:31:56.211140,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.299316e+09,Data Analyst / Data Engineer – SCG Indonesia,SCG,"Jakarta, Indonesia (On-site)",About the job\n\nAbout SCG in Indonesia\n\nWit...,https://www.linkedin.com/jobs/view/4299315688/...,Web Developer,Indonesia,"['power bi', 'r', 'tableau', 'python', 'sql', ...",6,2026-04-18T20:32:01.140377,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
import ast

def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

print(f"Data setelah filter: {len(df)} rows")
print(f"Skills masih kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['search_role'].value_counts())

Data setelah filter: 17864 rows
Skills masih kosong: 46 rows
search_role
Data Engineer           2376
Web Developer           1713
System Analyst          1685
Data Scientist          1491
Software Engineer       1351
Backend Developer       1005
Developer               1000
Full stack Developer    1000
Software Developer      1000
Data Analyst             978
Software                 885
IT Support               770
Frontend Developer       702
IT                       571
DevOps Engineer          405
Business Analyst         294
Full Stack Developer     112
Fullstack Developer      110
Programmer               108
Ai Engineer               83
ML Engineer               83
software                  64
Business Development      56
Other                     22
Name: count, dtype: int64


In [16]:
min_samples = 1000
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Data Engineer           2376
Web Developer           1713
System Analyst          1685
Data Scientist          1491
Software Engineer       1351
Backend Developer       1005
Full stack Developer    1000
Developer               1000
Software Developer      1000
Name: count, dtype: int64


In [18]:
max_samples = 1200
dfs = []
for role, group in df.groupby('search_role'):
    if len(group) > max_samples:
        group = resample(group, n_samples=max_samples, random_state=42)
    dfs.append(group)

df = pd.concat(dfs).reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Data Engineer           1200
Web Developer           1200
Data Scientist          1200
System Analyst          1200
Software Engineer       1200
Backend Developer       1005
Developer               1000
Full stack Developer    1000
Software Developer      1000
Name: count, dtype: int64


In [21]:
df['text_input'] = df['job_description'] + ' ' + df['extracted_skills_clean']

df = df[df['search_role'] != 'Software Engineer']
df = df[df['search_role'] != 'Web Developer']
df = df[df['search_role'] != 'Developer']
df = df[df['search_role'] != 'Software Developer']

print(df['search_role'].value_counts())

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

df['text_input'] = df['text_input'].fillna('').astype(str)
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

max_vocab_length = 20000
max_sequence_length = 512

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

search_role
Data Engineer           1200
Data Scientist          1200
System Analyst          1198
Backend Developer       1005
Full stack Developer    1000
Name: count, dtype: int64
Total role unik: 5
Mapping kelas:
{'Backend Developer': 0, 'Data Engineer': 1, 'Data Scientist': 2, 'Full stack Developer': 3, 'System Analyst': 4}

Shape X_train: (4482,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[   27     4    19  5126  5040  3568  5127  4820 15532 18414 18415 15577
  15594 15559 15545 15572 15717 15734     1 15776 15490 15560 15693 15501
    605 16525     1 15556  5104 15582 15523  5098  5131 16444  9493 15719
   5880 15713 17754   676     1 15576   605 16524  4026   190 17203   605
   1473 19585 16312 15660 15558 15489 15668  5130   977  1055  9112  6950
    118  4342   676  2271  6950   977  3277  2049    83  3275  1850  5627
  15721 15711 15476 19691 15694 15515 15546     1  4036  4040 19462  4035
   3517  5107  5079  4037  4473  8119  4043  7271  7688  7689  7690  403

In [22]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.core.series.Series'> object
4220    About the job\n\nエントリーを検討する際に役立つ情報をまとめた Entran...
1656    About The Company:\n\nThe working venue is in ...
4365    About the job\n\nPosition: eCommerce Developer...
Name: text_input, dtype: object
object


In [23]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3):
        super(CustomEarlyStopping, self).__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Overfitting terdeteksi. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

In [26]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, min_lr=0.001, verbose=1),
    CustomEarlyStopping(patience=3)
]

In [ ]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=128)(x)
x = Bidirectional(LSTM(64))(x)
x = Dropout(0.4)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier_LSTM")

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMulai proses training...")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[callbacks]
)



Mulai proses training...
Epoch 1/40
71/71 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.3751 - loss: 1.3982 - val_accuracy: 0.5049 - val_loss: 1.1858 - learning_rate: 0.0010
Epoch 2/40
71/71 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.5477 - loss: 1.0562 - val_accuracy: 0.5781 - val_loss: 0.9785 - learning_rate: 0.0010
Epoch 3/40
71/71 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.6606 - loss: 0.8008 - val_accuracy: 0.6262 - val_loss: 0.8271 - learning_rate: 0.0010
Epoch 4/40
71/71 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.7178 - loss: 0.6607 - val_accuracy: 0.6200 - val_loss: 0.8251 - learning_rate: 0.0010
Epoch 5/40
71/71 ━━━━━━━━━━━━━━━━━━━━ 89s 1s/step - accuracy: 0.7077 - loss: 0.6625 - val_accuracy: 0.6298 - val_loss: 0.8043 - learning_rate: 0.0010
Epoch 6/40


In [56]:
model_path = 'job_role_model.keras'
model.save(model_path)
print(f"[INFO] Model berhasil diekspor ke: {model_path}")

loaded_model = tf.keras.models.load_model(model_path)

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)

    pred_probs = loaded_model.predict(input_tensor, verbose=0)

    pred_class_idx = np.argmax(pred_probs)

    predicted_role = encoder.inverse_transform([pred_class_idx])[0]

    return predicted_role

skill_input = "i have experience for applying AI in machine learning, deep learning, Tensorflow, Python, NLP. Understanding of functional design principles, object-oriented programming principles, basic algorithms. ⁠Expertise in REST API development, NoSQL design, RDBMS design"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil diekspor ke: job_role_model.keras

--- Hasil Inference ---
Skill User : i have experience for applying AI in machine learning, deep learning, Tensorflow, Python, NLP. Understanding of functional design principles, object-oriented programming principles, basic algorithms. ⁠Expertise in REST API development, NoSQL design, RDBMS design
Cocok untuk: Backend Developer
